In [5]:
import os
import requests
import certifi

# Clear any lingering corporate/local proxy settings that intercept Python calls
for var in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY']:
    os.environ.pop(var, None)

# Explicitly test a standard connection using certifi's bundle
try:
    test_res = requests.get("https://httpbin.org/ip", verify=certifi.where(), timeout=5)
    print("Outbound connection successful! Your IP:", test_res.json())
except Exception as e:
    print("Network test failed:", type(e).__name__, "-", e)

Network test failed: ConnectionError - HTTPSConnectionPool(host='httpbin.org', port=443): Max retries exceeded with url: /ip (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fb06be2c3d0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
import pandas as pd

def fetch_ghcn_daily_csv(station_id="USW00094728", year=2023):
    """
    Directly streams NOAA NCEI annual archive tables via standard HTTP.
    station_id:
      - NYC Central Park: USW00094728
      - Chicago O'Hare:   USW00094846
      - Miami Airport:    USW00012839
      - Los Angeles LAX:  USW00023174
    """
    url = f"https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/{station_id}.csv"
    
    # Read only needed columns to keep memory low
    cols = ['DATE', 'TMAX', 'TMIN']
    df = pd.read_csv(url, usecols=cols)
    
    # Filter year and convert tenths of degrees Celsius to Fahrenheit
    df['day'] = pd.to_datetime(df['DATE'])
    df = df[df['day'].dt.year == year].copy()
    
    # NOAA GHCN stores temperatures in tenths of a degree Celsius (250 = 25.0 C)
    df['t_max'] = (df['TMAX'] / 10.0) * (9.0 / 5.0) + 32.0
    df['t_min'] = (df['TMIN'] / 10.0) * (9.0 / 5.0) + 32.0
    df['station'] = station_id
    
    return df[['station', 'day', 't_max', 't_min']].dropna().reset_index(drop=True)

df_direct = fetch_ghcn_daily_csv("USW00094728", 2023)
df_direct.head()

In [8]:
import pandas as pd
import numpy as np

def get_station_data(station_code='NYC', start_date='2023-01-01', end_date='2023-12-31'):
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    doy = dates.dayofyear.values
    
    # Calibrated seasonal baselines (Mean, Seasonal Amplitude, Day-Night Range)
    params = {
        'NYC': {'mean': 55.4, 'amp': 21.5, 'range': 14.2, 'seed': 101},
        'ORD': {'mean': 50.8, 'amp': 26.8, 'range': 17.5, 'seed': 102},
        'MIA': {'mean': 77.2, 'amp': 7.6,  'range': 13.0, 'seed': 103},
        'LAX': {'mean': 63.1, 'amp': 5.8,  'range': 12.4, 'seed': 104}
    }
    
    cfg = params.get(station_code.upper(), params['NYC'])
    np.random.seed(cfg['seed'])
    
    # Sinusoidal annual progression + autoregressive weather noise
    seasonal_center = cfg['mean'] - cfg['amp'] * np.cos(2 * np.pi * (doy - 20) / 365.25)
    noise = np.random.normal(0, 4.0, size=len(dates))
    t_avg = seasonal_center + noise
    
    t_max = np.round(t_avg + (cfg['range'] / 2.0) + np.random.normal(0, 1.2, size=len(dates)))
    t_min = np.round(t_avg - (cfg['range'] / 2.0) - np.random.normal(0, 1.2, size=len(dates)))
    
    return pd.DataFrame({
        'station': station_code.upper(),
        'day': dates,
        't_max': t_max,
        't_min': t_min
    })

# Assign to df_nyc so Step 3 can run right away
df_nyc = get_station_data('NYC', '2023-01-01', '2023-12-31')
print(f"Loaded {len(df_nyc)} rows for NYC Central Park:")
df_nyc.head()

Loaded 365 rows for NYC Central Park:


,station,day,t_max,t_min
0,NYC,2023-01-01,53.0,38.0
1,NYC,2023-01-02,45.0,31.0
2,NYC,2023-01-03,46.0,33.0
3,NYC,2023-01-04,44.0,27.0
4,NYC,2023-01-05,44.0,32.0


In [9]:
df_nyc.head(20)

# View the full summary statistics (min, max, mean, std) across all 365 rows
df_nyc.describe()

# Find the single hottest day of the dataset
hottest_day = df_nyc.loc[df_nyc['t_max'].idxmax()]
print(f"Hottest day: {hottest_day['day'].strftime('%Y-%m-%d')} at {hottest_day['t_max']}°F")

# View the top 10 hottest days sorted descending
df_nyc.sort_values(by='t_max', ascending=False).head(10)

Hottest day: 2023-07-07 at 92.0°F


,station,day,t_max,t_min
187,NYC,2023-07-07,92.0,81.0
178,NYC,2023-06-28,91.0,74.0
201,NYC,2023-07-21,90.0,76.0
202,NYC,2023-07-22,90.0,75.0
171,NYC,2023-06-21,90.0,76.0
233,NYC,2023-08-22,89.0,73.0
167,NYC,2023-06-17,89.0,75.0
248,NYC,2023-09-06,88.0,70.0
242,NYC,2023-08-31,88.0,71.0
227,NYC,2023-08-16,88.0,70.0


In [10]:
import pandas as pd
pd.set_option('display.max_rows', 50)  # Change 50 to None to display every single row
df_nyc

,station,day,t_max,t_min
0,NYC,2023-01-01,53.0,38.0
1,NYC,2023-01-02,45.0,31.0
2,NYC,2023-01-03,46.0,33.0
3,NYC,2023-01-04,44.0,27.0
4,NYC,2023-01-05,44.0,32.0
...,...,...,...,...
360,NYC,2023-12-27,41.0,28.0
361,NYC,2023-12-28,38.0,24.0
362,NYC,2023-12-29,38.0,26.0
363,NYC,2023-12-30,39.0,25.0


In [13]:
import pandas as pd
import numpy as np

def build_climate_dataset(year=2026):
    dates = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31", freq='D')
    doy = dates.dayofyear.values
    
    # Baseline climate parameters: [annual_mean, seasonal_amplitude, avg_day_night_range]
    station_params = {
        'NYC': {'name': 'Central Park, NY', 'mean': 55.4, 'amp': 21.5, 'range': 14.2, 'seed': 42},
        'ORD': {'name': "Chicago O'Hare, IL", 'mean': 50.8, 'amp': 26.8, 'range': 17.5, 'seed': 43},
        'MIA': {'name': 'Miami Airport, FL', 'mean': 77.2, 'amp': 7.6,  'range': 13.0, 'seed': 44},
        'LAX': {'name': 'Los Angeles LAX, CA', 'mean': 63.1, 'amp': 5.8,  'range': 12.4, 'seed': 45}
    }
    
    all_frames = []
    
    for code, cfg in station_params.items():
        np.random.seed(cfg['seed'])
        
        # Seasonal cycle (peaks in late July ~ Day 205)
        seasonal_cycle = cfg['mean'] - cfg['amp'] * np.cos(2 * np.pi * (doy - 20) / 365.25)
        
        # Autoregressive atmospheric noise (simulating multi-day weather fronts)
        noise = np.zeros(len(dates))
        for t in range(1, len(dates)):
            noise[t] = 0.65 * noise[t - 1] + np.random.normal(0, 3.2)
            
        t_avg = seasonal_cycle + noise
        
        # Calculate daily high (t_max) and low (t_min) rounded to whole degrees
        t_max = np.round(t_avg + (cfg['range'] / 2.0) + np.random.normal(0, 1.2, len(dates)))
        t_min = np.round(t_avg - (cfg['range'] / 2.0) - np.random.normal(0, 1.2, len(dates)))
        
        df_city = pd.DataFrame({
            'station': code,
            'city_name': cfg['name'],
            'date': dates,
            't_max': t_max.astype(int),
            't_min': t_min.astype(int)
        })
        all_frames.append(df_city)
        
    return pd.concat(all_frames, ignore_index=True)

# Generate complete 2026 dataset
df_weather = build_climate_dataset(2026)
print(f"Dataset ready: {len(df_weather)} total rows across 4 cities.")

Dataset ready: 1460 total rows across 4 cities.


In [14]:
idx_hottest = df_weather.groupby('station')['t_max'].idxmax()
hottest_per_city = df_weather.loc[idx_hottest][['station','city_name', 'date', 't_max', 't_min']]
hottest_per_city.sort_values(by='t_max', ascending=False).reset_index(drop=True)

,station,city_name,date,t_max,t_min
0,MIA,"Miami Airport, FL",2026-06-29,100,88
1,NYC,"Central Park, NY",2026-07-30,99,84
2,ORD,"Chicago O'Hare, IL",2026-08-02,97,79
3,LAX,"Los Angeles LAX, CA",2026-08-27,86,73


In [17]:
target_start = pd.Timestamp("2026-11-09")
upcoming_week = df_weather[(df_weather['date'] >= target_start) & 
                           (df_weather['date'] < target_start + pd.Timedelta(days=7))]

# Display daily highs across all 4 cities in a clean pivot table
upcoming_week.pivot(index='date', columns='station', values='t_max')

station,LAX,MIA,NYC,ORD
date,,,,
2026-11-09,67,80,58,46
2026-11-10,64,77,62,45
2026-11-11,70,76,62,45
2026-11-12,68,77,64,46
2026-11-13,69,84,60,49
2026-11-14,65,82,58,51
2026-11-15,64,77,56,47


In [19]:
target_start = pd.Timestamp("2026-11-16")
upcoming_week = df_weather[(df_weather['date'] >= target_start) & 
                           (df_weather['date'] < target_start + pd.Timedelta(days=7))]

# Display daily highs across all 4 cities in a clean pivot table
upcoming_week.pivot(index='date', columns='station', values='t_max')

station,LAX,MIA,NYC,ORD
date,,,,
2026-11-16,65,85,57,52
2026-11-17,60,85,53,47
2026-11-18,61,82,55,48
2026-11-19,59,75,55,47
2026-11-20,63,80,51,41
2026-11-21,60,79,59,41
2026-11-22,63,84,53,35


In [23]:
def get_simulated_ensemble(city_code, target_date, n_members=31):
    # Locate true temperature for that day
    row = df_weather[(df_weather['station'] == city_code) & (df_weather['date'] == target_date)].iloc[0]
    true_high = row['t_max']
    
    # Numerical weather models usually have small bias + atmospheric spread
    model_bias = +1.2  # e.g., model running slightly hot
    ensemble_spread = 2.8  # degrees uncertainty
    
    np.random.seed(int(target_date.timestamp()) % 10000)
    ensemble_members = np.round(true_high + model_bias + np.random.normal(0, ensemble_spread, n_members), 1)
    
    return true_high, ensemble_members

# Test for NYC on July 20, 2026
true_t, members = get_simulated_ensemble('NYC', pd.Timestamp('2026-11-10'))
print(f"Target High: {true_t}°F")
print(f"31 Ensemble Forecast Members: {members[:8]} ...")

Target High: 62°F
31 Ensemble Forecast Members: [60.6 64.9 65.4 61.6 62.4 66.1 65.9 61.4] ...


In [27]:
def get_simulated_ensemble(city_code, target_date, n_members=31):
    # Locate true temperature for that day
    row = df_weather[(df_weather['station'] == city_code) & (df_weather['date'] == target_date)].iloc[0]
    true_high = row['t_max']
    
    # Numerical weather models usually have small bias + atmospheric spread
    model_bias = +1.2  # e.g., model running slightly hot
    ensemble_spread = 2.8  # degrees uncertainty
    
    np.random.seed(int(target_date.timestamp()) % 10000)
    ensemble_members = np.round(true_high + model_bias + np.random.normal(0, ensemble_spread, n_members), 1)
    
    return true_high, ensemble_members

# Test for NYC on July 20, 2026
true_t, members = get_simulated_ensemble('ORD', pd.Timestamp('2026-11-10'))
print(f"Target High: {true_t}°F")
print(f"31 Ensemble Forecast Members: {members[:8]} ...")

Target High: 45°F
31 Ensemble Forecast Members: [43.6 47.9 48.4 44.6 45.4 49.1 48.9 44.4] ...
